# Calibrated Explanations: Plotly Uncertainty Quadrant

This notebook demonstrates the `plotly.local.uncertainty_quadrant` visualization plugin
using the standard `explanation.plot(style=...)` API.

Each point in the chart represents one factual CE rule or feature contribution:

- **x-axis**: signed contribution / feature weight
- **y-axis**: contribution interval width (`high - low`)
- **colour**: deterministic quadrant label

Install this package and Plotly rendering support with:

```bash
pip install calibrated-explanations-visualization-plotly[plotly]
```

In [2]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd()
if package_dir.name == "examples":
    package_dir = package_dir.parent
else:
    repo_candidate = Path("packages/visualization/calibrated-explanations-visualization-plotly")
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", f"{package_dir}[plotly]"]
)

0

## Classification Example

The dataset is split into three parts:

- **proper training set** — used to fit the model
- **calibration set** — used to calibrate the conformal predictor
- **test set** — instances to explain

The model is fitted and calibrated via `WrapCalibratedExplainer`, then one test
instance is explained and rendered with `explanation.plot(style=STYLE_ID, ...)`.

In [3]:
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import STYLE_ID
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

STYLE_ID

'plotly.local.uncertainty_quadrant'

In [4]:
X, y = make_classification(
    n_samples=160,
    n_features=20,
    n_informative=4,
    n_redundant=0,
    random_state=7,
)
X_train, X_test, y_train, _ = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)
X_train_proper, X_cal, y_train_proper, y_cal = train_test_split(
    X_train, y_train, test_size=0.25, random_state=7, stratify=y_train
)

explainer = WrapCalibratedExplainer(
    LogisticRegression(solver="liblinear", random_state=7)
).fit(X_train_proper, y_train_proper)
explainer.calibrate(X_cal, y_cal)
classification_explanations = explainer.explain_factual(X_test[:1])
classification_explanations.explanations[0]

c:\Users\loftuw\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py:126: UserWarning: Cache backend fallback: using minimal in-package LRU/TTL implementation due to missing 'cachetools'
  return _bootstrap._gcd_import(name[level:], package, level)


Prediction [ Low ,  High]
0.714 [0.667, 0.833]
Value : Feature                                  Weight [ Low  ,  High ]
0.78  : 13 > -0.37                                0.318 [ 0.187,  0.548]
0.61  : 10 > 0.29                                -0.109 [-0.286, -0.071]
-0.94 : 3 <= 1.75                                 0.089 [-0.119,  0.214]
-0.89 : 8 <= 1.58                                 0.089 [-0.119,  0.214]
0.15  : 15 <= 0.72                               -0.045 [-0.286,  0.032]
-0.6  : 4 <= 0.49                                -0.036 [-0.286,  0.048]
-0.44 : 7 > -1.58                                -0.036 [-0.286,  0.048]
0.89  : 2 > 0.29                                  0.021 [-0.119,  0.085]

Render the uncertainty quadrant by passing `style=STYLE_ID` to `explanation.plot`.
All layout options (`filter_top`, `sort_by`, etc.) are forwarded as keyword arguments.

In [5]:
result = classification_explanations.plot(
    style=STYLE_ID,
)
result.extras["figure"]

## Threshold Overrides

By default the effect threshold is the median absolute contribution and the width
threshold is the median interval width. Both can be overridden for domain-specific
definitions of "large" and "wide".

In [6]:
result_override = classification_explanations.plot(
    style=STYLE_ID,
    show=False,
    effect_threshold=0.05,
    width_threshold=0.25,
    sort_by="status",
)
result_override.extras["figure"]

## Regression Example

The same three-way split is used: proper training, calibration, and test.
The model is fitted and calibrated before explaining a test instance.
The x-axis remains the signed contribution and the y-axis the interval width.

In [7]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

X_reg, y_reg = make_regression(
    n_samples=160, n_features=5, noise=0.3, random_state=11
)
X_reg_train, X_reg_test, y_reg_train, _ = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=11
)
X_reg_proper, X_reg_cal, y_reg_proper, y_reg_cal = train_test_split(
    X_reg_train, y_reg_train, test_size=0.25, random_state=11
)

reg_explainer = WrapCalibratedExplainer(
    LinearRegression()
).fit(X_reg_proper, y_reg_proper)
reg_explainer.calibrate(X_reg_cal, y_reg_cal)
regression_explanations = reg_explainer.explain_factual(X_reg_test[:1])

In [8]:
result_reg = regression_explanations.plot(
    style=STYLE_ID,
)
result_reg.extras["figure"]

## Exporting to HTML

Pass a file path to save an interactive Plotly HTML file alongside the inline figure:

In [9]:
result_reg_saved = regression_explanations.plot(
    style=STYLE_ID,
    path="uncertainty_quadrant_regression.html",
)
print("Saved to:", result_reg_saved.saved_paths)

Saved to: ()


In [10]:
regression_explanations = reg_explainer.explain_factual(X_reg_test[:1], threshold=y_reg_cal[0])
regression_explanations.plot(
    style=STYLE_ID,
)

PlotRenderResult(artifact={'artifact_type': 'plotly.local.uncertainty_quadrant', 'style': 'plotly.local.uncertainty_quadrant', 'prediction': {'prediction': 0.8571428571428572, 'low': 0.8333333333333334, 'high': 1.0, 'classes': 1.0}, 'items': [{'index': 0, 'rule_label': '0 <= -1.44', 'feature': 0, 'feature_name': '0', 'instance_value': -1.8485851893870515, 'contribution': -0.03412698412698412, 'low': -0.1428571428571428, 'high': -0.019961519961519958, 'interval_width': 0.12289562289562284, 'status_label': 'robust_driver'}, {'index': 1, 'rule_label': '1 > -0.51', 'feature': 1, 'feature_name': '1', 'instance_value': -0.217903410574504, 'contribution': -0.019961519961519958, 'low': -0.1428571428571428, 'high': -0.0011904761904761862, 'interval_width': 0.1416666666666666, 'status_label': 'robust_driver'}, {'index': 4, 'rule_label': '4 <= 0.58', 'feature': 4, 'feature_name': '4', 'instance_value': -0.23331191858949996, 'contribution': -0.015331890331890308, 'low': -0.1428571428571428, 'high'